# GBD-style data pipeline template

This notebook demonstrates a lightweight workflow for turning a GBD-style country table into browser-friendly GeoJSON and a GeoLibre project file.

**Bundled default:** artificial demo data.

**Real workflow:** download a CSV from IHME/GHDx/GBD Results, place it at `data/ihme_gbd_results.csv`, and re-run this notebook. Keep only public, aggregated data suitable for GitHub Pages. Do not publish sensitive, private, or restricted data.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("data")
PROJECT_DIR = Path("projects")
user_file = DATA_DIR / "ihme_gbd_results.csv"
sample_file = DATA_DIR / "sample_gbd_like_burden.csv"
input_file = user_file if user_file.exists() else sample_file

print("Using:", input_file)
df = pd.read_csv(input_file)
df.head()

In [ ]:
REQUIRED_COLUMNS = {
    "location_name", "year", "measure_name", "metric_name", "age_name", "sex_name", "cause_name", "val"
}

def normalize_gbd_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize common GBD export column variants used by IHME/GHDx downloads."""
    out = frame.copy()
    out.columns = [c.strip() for c in out.columns]
    # Some exports use lowercase already; this keeps the demo forgiving.
    lower_map = {c.lower(): c for c in out.columns}
    renames = {}
    canonical = ["location_name", "year", "measure_name", "metric_name", "age_name", "sex_name", "cause_name", "val", "lower", "upper"]
    for c in canonical:
        if c not in out.columns and c in lower_map:
            renames[lower_map[c]] = c
    out = out.rename(columns=renames)
    missing = REQUIRED_COLUMNS - set(out.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return out

def filter_for_lite(frame: pd.DataFrame,
                    measure="DALYs",
                    metric="Rate",
                    age="Age-standardized",
                    sex="Both",
                    years=None,
                    causes=None) -> pd.DataFrame:
    """Reduce the table to a small subset suitable for JupyterLite and GitHub Pages."""
    out = frame.copy()
    if measure is not None:
        out = out[out["measure_name"].astype(str).str.lower() == measure.lower()]
    if metric is not None:
        out = out[out["metric_name"].astype(str).str.lower() == metric.lower()]
    if age is not None:
        out = out[out["age_name"].astype(str).str.lower() == age.lower()]
    if sex is not None:
        out = out[out["sex_name"].astype(str).str.lower() == sex.lower()]
    if years is not None:
        out = out[out["year"].isin(years)]
    if causes is not None:
        out = out[out["cause_name"].isin(causes)]
    return out

clean = normalize_gbd_columns(df)
lite = filter_for_lite(clean, years=[clean["year"].max()])
print(f"Rows after Lite filter: {len(lite):,}")
lite.head()

In [ ]:
# Join to a lightweight country reference table with centroids.
# For production, replace this table with a maintained ISO3 boundary/centroid source.
countries = pd.read_csv(DATA_DIR / "country_reference.csv")
joined = lite.merge(countries[["iso3", "longitude", "latitude"]], on="iso3", how="left") if "iso3" in lite.columns else lite.merge(countries, on="location_name", how="left")
missing_geo = joined[joined["longitude"].isna() | joined["latitude"].isna()]
if len(missing_geo):
    print("Warning: missing country coordinates for:")
    display(missing_geo[["location_name"]].drop_duplicates().head(20))
joined = joined.dropna(subset=["longitude", "latitude"]).copy()
joined.head()

In [ ]:
def to_geojson(frame: pd.DataFrame) -> dict:
    features = []
    for row in frame.to_dict(orient="records"):
        lon = float(row.pop("longitude"))
        lat = float(row.pop("latitude"))
        # Convert numpy-ish values to JSON-friendly values.
        props = {k: (None if pd.isna(v) else v) for k, v in row.items()}
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lon, lat]},
            "properties": props,
        })
    return {"type": "FeatureCollection", "features": features}

geojson = to_geojson(joined)
out_geojson = DATA_DIR / "gbd_country_points_generated.geojson"
with out_geojson.open("w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)
print("Wrote", out_geojson)
print("Features:", len(geojson["features"]))

In [ ]:
def make_geolibre_project(geojson: dict, title="Generated Global Health Demo") -> dict:
    return {
        "version": "0.1.0",
        "name": title,
        "mapView": {"center": [8.0, 12.0], "zoom": 1.15, "bearing": 0, "pitch": 0},
        "basemapStyleUrl": "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json",
        "basemapVisible": True,
        "basemapOpacity": 1,
        "layers": [{
            "id": "generated_gbd_points",
            "name": "Generated GBD-style points",
            "type": "geojson",
            "source": {"type": "geojson"},
            "visible": True,
            "opacity": 0.85,
            "style": {
                "minZoom": 0,
                "maxZoom": 24,
                "circleRadius": 7,
                "fillColor": "#ef4444",
                "strokeColor": "#111827",
                "strokeWidth": 1,
                "strokeWidthUnit": "pixels",
                "fillOpacity": 0.72,
                "rasterBrightnessMin": 0,
                "rasterBrightnessMax": 1,
                "rasterSaturation": 0,
                "rasterContrast": 0,
                "rasterHueRotate": 0,
            },
            "metadata": {"data_note": "Verify source, license, and citation before publication."},
            "geojson": geojson,
        }],
        "styles": {},
        "metadata": {
            "data_note": "Generated from a GBD-style table. Check IHME/GHDx user agreement and citation requirements before publishing real data.",
        },
    }

project = make_geolibre_project(geojson)
PROJECT_DIR.mkdir(exist_ok=True)
out_project = PROJECT_DIR / "generated_global_health_demo.geolibre.json"
with out_project.open("w", encoding="utf-8") as f:
    json.dump(project, f, indent=2)
print("Wrote", out_project)

## Next step

Open `02_jupytergis_global_health_points.ipynb` to visualize the generated GeoJSON with JupyterGIS. Open `03_geolibre_project_launcher.ipynb` to launch the `.geolibre.json` project in GeoLibre Web.